# Your First Vector RAG Application: Cat Health Assistant

In this notebook, we will build a dense vector retrieval application using **LangChain v1**, **OpenAI embeddings**, and **Qdrant** as an in-memory vector database.

The goal is to understand the core RAG loop:

1. Load a cat health guideline PDF
2. Split it into smaller chunks
3. Embed those chunks
4. Store the embeddings in Qdrant
5. Retrieve relevant chunks for a question
6. Generate an answer grounded in the retrieved context

> Note: This notebook expects Python 3.12 and uses uv for dependency management.

> Note: This is a vector RAG lesson, not a veterinary care tool. The assistant should answer from the PDF and point users to a veterinarian for diagnosis, treatment, medication, or urgent care decisions.

## Table of Contents

- Task 1: Environment Setup
- Task 2: Embedding Similarity Primer
- Task 3: Documents - Loading the Cat Health Guideline PDF
- Task 4: Chunking the Documents
- Task 5: Embeddings and Qdrant
- Task 6: Retrieval with Scores
- Task 7: Retrieval Augmented Generation
- Activity: Tune Retrieval Quality

## Task 1: Environment Setup

From the `01_Dense_Vector_Retrieval` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_openai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [ ]:
from pathlib import Path
from math import sqrt
from getpass import getpass
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

### OpenAI API Key

The chat model and embedding model both use OpenAI. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Task 2: Embedding Similarity Primer

Before we load a full PDF, let's make dense vector retrieval less mysterious.

An embedding model turns text into a list of numbers. Texts with related meaning should land closer together in that vector space.

A common way to score closeness is **cosine similarity**:

```text
cosine_similarity(a, b) = dot_product(a, b) / (length(a) * length(b))
```

The intuition: if two vectors point in a similar direction, their cosine similarity is higher. Vector databases like Qdrant use this same idea, but at a much larger scale.

In [ ]:
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embedding_model)

example_texts = [
    "king",
    "queen",
    "banana",
    "cat",
    "veterinarian",
    "cat health guidelines",
]

example_vectors = dict(zip(example_texts, embeddings.embed_documents(example_texts)))


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    length_a = sqrt(sum(a * a for a in vector_a))
    length_b = sqrt(sum(b * b for b in vector_b))
    return dot_product / (length_a * length_b)


comparison_pairs = [
    ("king", "queen"),
    ("king", "banana"),
    ("cat", "veterinarian"),
    ("cat", "cat health guidelines"),
]

for left, right in comparison_pairs:
    score = cosine_similarity(example_vectors[left], example_vectors[right])
    print(f"{left:>22} <> {right:<22} score={score:.3f}")

A few important notes:

- The score is useful for ranking, not as an absolute truth about meaning.
- Different embedding models can produce different scores.
- In RAG, we embed each document chunk once, then embed the user's query and search for the nearest chunk vectors.

That is the retrieval part of RAG.

## Task 3: Documents

LangChain represents loaded text as `Document` objects. A `Document` has:

- `page_content`: the text
- `metadata`: information such as source file and page number

We will load one `Document` per PDF page, then split those pages into smaller chunks.

### Course PDF

This notebook uses the bundled cat health guideline PDF at:

```text
01_Dense_Vector_Retrieval/data/cat_health_guidelines.pdf
```

The next cell checks that the course material is present before we start loading pages.

In [ ]:
pdf_path = Path("data/cat_health_guidelines.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(
        f"Expected the cat health guideline PDF at: {pdf_path.resolve()}\n"
        "The bundled course PDF is missing from this copy of the materials."
    )

### Load the PDF

`PyPDFLoader` extracts text from text-based PDFs. If the PDF is scanned images, this loader may return little or no text, and OCR would be needed.

In [ ]:
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()

for page in pages:
    page.metadata["source"] = pdf_path.name
    page.metadata["document_type"] = "cat_health_guideline"

pages = [page for page in pages if page.page_content.strip()]

if not pages:
    raise ValueError(
        "The PDF loaded, but no extractable text was found. "
        "This usually means the PDF is scanned and needs OCR first."
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")

In [ ]:
print(pages[0].page_content[:750])
print("\nMetadata:", pages[0].metadata)

#### ❓Question #1

Why is metadata important for a RAG application?

##### ✅ Answer:

Metadata lets us trace where retrieved content came from (page number, source document), enables source filtering before retrieval, and helpfull for debugging poor retrieval results.

## Task 4: Chunking the Documents

A full PDF page can be too large or too mixed-topic for high-quality retrieval. We split pages into overlapping chunks so each chunk has enough local context but is still focused.

Here we will start with chunks of 1,000 characters and 200 characters of overlap. The chunk size controls how much text each vector represents; the overlap keeps nearby context from being lost at chunk boundaries.

`RecursiveCharacterTextSplitter` tries to split on natural boundaries first, such as paragraphs and line breaks, before falling back to smaller separators.

In [ ]:
chunk_size = 1500
chunk_overlap = 300

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
)

splits = text_splitter.split_documents(pages)

print(f"Split {len(pages)} pages into {len(splits)} chunks.")
print(f"Chunk size: {chunk_size} characters")
print(f"Chunk overlap: {chunk_overlap} characters")

In [ ]:
sample_chunk = splits[0]
print(sample_chunk.page_content[:750])
print("\nMetadata:", sample_chunk.metadata)

#### ❓Question #2

What tradeoff do we make when choosing chunk size and chunk overlap?

##### ✅ Answer:

Larger chunks give more context per retrieval but reduce precision as irrelevant content gets included. Smaller chunks are more precise but may cut off important context. Overlap reduces the risk of splitting a key sentence across chunk boundaries, but increases storage and computation.As this document and app is related to medical where context determines accuracy so after expriment I am going with 1500 / 300 .

Use the [Chunk Visualizer](https://chunkviz.up.railway.app/) to experiment with different chunk sizes and overlaps and see how the text boundaries change.

## Task 5: Embeddings and Qdrant

Now we apply the same embedding idea to every chunk from the PDF. Qdrant stores those vectors and lets us search for chunks that are close to a query in embedding space.

We already created an OpenAI embedding model in the primer above. The Qdrant collection name is just a label for the set of vectors we are creating.

For this notebook, Qdrant runs in memory with `location=":memory:"`. That means no Docker, no Qdrant Cloud account, and no persistence after the notebook kernel stops.

In [ ]:
collection_name = "cat_health_guidelines"

vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
    force_recreate=True,
)

print(f"Embedded chunks with: {embedding_model}")
print(f"Built in-memory Qdrant collection: {collection_name}")

## Task 6: Retrieval with Scores

Before we generate answers, we should inspect retrieval directly. If retrieval returns poor context, the final answer will usually be poor too.

The value `k` controls how many chunks the retriever returns. A larger `k` gives the model more context, but it can also add noise. We will start with `k = 4` and tune it later.

Qdrant can return both the matching `Document` and a similarity score. This is the same ranking idea we saw with `king`, `queen`, and `cat`, now applied to PDF chunks.

In [ ]:
def display_retrieval_results(query: str, k: int) -> list[tuple]:
    """Retrieve chunks and print a compact view of the results."""
    results = vector_store.similarity_search_with_score(query, k=k)

    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)

    return results

In [ ]:
retrieval_k = 4
retrieval_query = "What signs suggest that a cat should be seen by a veterinarian?"
retrieved_results = display_retrieval_results(retrieval_query, k=retrieval_k)

#### ❓Question #3

What does a similarity score help you understand, and what does it not prove by itself?

##### ✅ Answer:

It tells us how semantically similar a chunk is to the query. It doesn't prove the chunk actually answers the question, a highly similar chunk might be thematically related but not contain the specific answer.

## Task 7: Retrieval Augmented Generation

Now we combine retrieval with generation. We will use a two-step RAG pattern:

1. Retrieve relevant chunks from Qdrant
2. Put those chunks into the prompt and ask the model to answer from the context

This is intentionally simpler than an agent. We always retrieve before answering, which makes the vector retrieval mechanics easy to inspect.

For generation, we will use `gpt-5.4-mini`.

In [ ]:
chat_model = "gpt-5.4-mini"
llm = ChatOpenAI(model=chat_model)

RAG_SYSTEM_PROMPT = """You are a cat health guideline assistant in a vector RAG lesson.

Use only the provided context to answer the user's question.
If the context does not contain enough information, say: "I don't have enough information in the provided cat health guideline PDF to answer that."

Cite the retrieved sources inline using labels like [Source 1] or [Source 2].
Do not diagnose, prescribe medication, or replace a veterinarian.
For diagnosis, treatment decisions, medication questions, or urgent symptoms, recommend contacting a veterinarian.
Keep the answer concise and practical."""

RAG_USER_PROMPT = """Context:
{context}

Question: {question}

Answer from the context above."""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

rag_chain = rag_prompt | llm | StrOutputParser()

In [ ]:
def format_context(scored_docs: list[tuple]) -> str:
    """Convert retrieved documents into a source-labeled context string."""
    formatted_chunks = []

    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        source = doc.metadata.get("source", "unknown source")

        formatted_chunks.append(
            f"[Source {index}] {source}, page {page_display}, score {score:.3f}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


def answer_question(question: str, k: int) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_docs = vector_store.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    answer = rag_chain.invoke({"context": context, "question": question})

    sources = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context": scored_docs,
    }

Before calling the model, inspect the formatted context. This is the exact text that will be inserted into the RAG prompt.

In [ ]:
example_context = format_context(retrieved_results[:2])
print(example_context[:2000])

In [ ]:
answer_k = 4

result = answer_question(
    "What are signs that my cat may need veterinary attention?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

### Vibe Check Queries

Run a few questions that should be answerable from a cat health guideline PDF. Then run one question that may not be answerable and confirm the assistant says it does not have enough information.

In [ ]:
vibe_check_questions = [
    "What preventive care is recommended for cats?",
    "What symptoms should make me call a veterinarian?",
    "What should I know about feeding a healthy adult cat?",
    "Can my cat help me file my taxes?",
]

for question in vibe_check_questions:
    print("Question:", question)
    print(answer_question(question, k=answer_k)["answer"])
    print("=" * 100)

#### ❓Question #4

For the vibe check queries above, did the retrieved context seem relevant before generation? Why or why not?

##### ✅ Answer:

For the first three questions — preventive care, symptoms, and feeding, the retrieved chunks were clearly relevant. They contained actual 
medical advice from the PDF that directly addressed each question, with 
scores consistently above 0.7.

For the last question about filing taxes, no relevant chunks came back 
and the assistant correctly said: "I don’t have enough information in the provided cat health guideline PDF to answer that."

## 🏗️ Activity: Tune Retrieval Quality

Improve retrieval quality by changing one or more of these values:

- The chunk size
- The chunk overlap
- The retrieval `k`
- The wording of the retrieval query

Suggested workflow:

1. Pick one test question.
2. Inspect the retrieved chunks and scores.
3. Change one retrieval setting.
4. Rebuild the splitter and vector store.
5. Compare whether the retrieved chunks became more relevant.

When you are done, write down what changed and whether the final answer improved.

In [ ]:
# Activity workspace
# Try retrieval tuning here.
test_query = "What should be included in a wellness exam for cats?"

# BEFORE: defaults 1000/200
text_splitter_before = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
splits_before = text_splitter_before.split_documents(pages)
vs_before = QdrantVectorStore.from_documents(
    documents=splits_before,
    embedding=embeddings,
    location=":memory:",
    collection_name="before",
    force_recreate=True,
)

print("=== BEFORE (1000/200) ===")
for i, (doc, score) in enumerate(vs_before.similarity_search_with_score(test_query, k=4), 1):
    print(f"Source {i} | score={score:.3f}")
    print(doc.page_content[:200])
    print()

# MIDDLE: 1200/250
text_splitter_middle = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=250,
    add_start_index=True,
)
splits_middle = text_splitter_middle.split_documents(pages)
vs_middle = QdrantVectorStore.from_documents(
    documents=splits_middle,
    embedding=embeddings,
    location=":memory:",
    collection_name="middle",
    force_recreate=True,
)

print("=== MIDDLE (1200/250) ===")
for i, (doc, score) in enumerate(vs_middle.similarity_search_with_score(test_query, k=4), 1):
    print(f"Source {i} | score={score:.3f}")
    print(doc.page_content[:200])
    print()

# AFTER: 1500/300
text_splitter_after = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=300,
    add_start_index=True,
)
splits_after = text_splitter_after.split_documents(pages)
vs_after = QdrantVectorStore.from_documents(
    documents=splits_after,
    embedding=embeddings,
    location=":memory:",
    collection_name="after",
    force_recreate=True,
)

print("=== AFTER (1500/300) ===")
for i, (doc, score) in enumerate(vs_after.similarity_search_with_score(test_query, k=4), 1):
    print(f"Source {i} | score={score:.3f}")
    print(doc.page_content[:200])
    print()

print("=== K EXPERIMENT ===")
for label, vs in [("1000/200", vs_before), ("1200/250", vs_middle), ("1500/300", vs_after)]:
    print(f"\n{'='*40}")
    print(f"Settings: {label}")
    for k in [2, 4, 6]:
        print(f"\n--- k={k} ---")
        results = vs.similarity_search_with_score(test_query, k=k)
        for i, (doc, score) in enumerate(results, 1):
            print(f"Source {i} | score={score:.3f}")
            print(doc.page_content[:200])
            print()

### 🏗️ Activity Notes
Query used: "What should be included in a wellness exam for cats?"

Settings changed: chunk_size (1000, 1200, 1500) and chunk_overlap 
(200, 250, 300) and retrieval k (2, 4, 6)

Before (1000/200) k=4:
Top scores: 0.649, 0.640, 0.627, 0.619
Source 1 retrieved content about cat exposure to other cats and 
boarding facilities. Source 2 retrieved signs of pain and anxiety 
in mature and senior cats. Source 3 retrieved medical history focus 
for new patients. Source 4 retrieved cost of care and pet insurance 
discussions. At k=6, scores dropped to 0.618 and 0.607 with 
kitten behavior content appearing alongside wellness exam content.

Middle (1200/250) k=4:
Top scores: 0.648, 0.642, 0.639, 0.631
Retrieved more clinically focused content — indoor/outdoor access 
details, handling preferences, disease detection for adult and 
senior cats, and subtle signs of anxiety and illness. At k=6 scores 
stayed strong at 0.629 and 0.627 meaning extra chunks remained 
relevant.

After (1500/300) k=4:
Top scores: 0.648, 0.641, 0.633, 0.626
Source 1 retrieved exercise tolerance alongside vomiting and hairball 
questions — clinically related topics kept together. Source 2 
retrieved the full lifestyle classification for preventive care 
recommendations as a complete thought. Source 3 retrieved 
temperament, demeanor, and handling preferences. Source 4 retrieved 
pain and anxiety evaluation across all life stages. At k=6 scores 
held at 0.620 and 0.617 with the human-cat bond and consultation 
agenda setting — both still relevant to a wellness exam.

Did retrieval improve? Why or why not?
Yes, with nuance. Similarity scores were close across all three 
settings with less than 0.02 difference. The improvement showed 
in two areas. First, content coherence — the 1000/200 setting 
retrieved cost of care and insurance discussions at k=4 which are 
not directly relevant to a wellness exam. The 1500/300 setting kept 
clinically related content together in the same chunk. Second, k 
stability — at k=6 the 1000/200 setting dropped to 0.607 with less 
relevant content while the 1500/300 setting held at 0.617 with 
chunks still relevant to the query. For this medical document 
1500/300 with k=4 gave the best balance of complete clinical context 
without adding noise from lower scoring sources.